# 26 · RAG 幻觉：定位与治理

> 幻觉不是“模型的错”，在 RAG 里它往往能被分解定位到具体环节。本课给一套定位与治理框架。

**本文件覆盖知识点**：Retrieval Error → Context 错 → Generation Error → Hallucination / Grounded Generation / Citation·Attribution / Faithfulness / Answer Verification

In [2]:
# ===== 本课共用：真调 LLM 做「说明 / 演示」的小助手 =====
# 凡某个知识点能靠“真调一次大模型”当场讲清 / 演示的，下面的 cell 都用 _llm_live()
# 真调 qwen-plus 并打印模型输出作为说明；只有在项目根 .env 配了 DASHSCOPE_API_KEY 时才真调，
# 没配置就打印一段固定的演示样例，保证整个 notebook 不联网也能完整读下来。
from dotenv import load_dotenv; load_dotenv()
import os
from dashscope import Generation

_KEY = os.getenv('DASHSCOPE_API_KEY', '').strip()
_HAS_KEY = bool(_KEY) and '你的' not in _KEY

def _llm_live(prompt, fallback, system='你是资深 RAG 讲师，回答精炼、结构清晰、尽量结合例子。', temperature=0.3, model='qwen-plus'):
    """真调一次 qwen-plus 并打印结果；无 Key 时打印 fallback 作为演示样例。返回模型文本或 None。"""
    if not _HAS_KEY:
        print('未在 .env 配置 DASHSCOPE_API_KEY，跳过实时调用。以下是固定演示样例（配置后自动变为实时输出）：')
        print(fallback)
        return None
    msgs = [{'role': 'system', 'content': system}, {'role': 'user', 'content': prompt}]
    try:
        r = Generation.call(model=model, messages=msgs, temperature=temperature, result_format='message', api_key=_KEY)
        if r.status_code == 200:
            text = r.output.choices[0].message.content
            print('—— 模型实时输出 ——')
            print(text)
            return text
        print('调用失败：', getattr(r, 'code', ''), getattr(r, 'message', ''))
    except Exception as e:
        print('调用异常：', e)
    print('fallback：')
    print(fallback)
    return None


## 1. 幻觉从哪来：三级定位

```text
Retrieval Error ──→ Context 错（根本没召回对的内容）
    Generation Error ──→ 模型没按资料说（召回了却没读懂/胡编）
              ↓
          Hallucination
```

| 现象 | 通常根因 | 对策方向 |
|------|---------|---------|
| 答案内容与事实不符 | 检索没召回正确片段 | 优化召回：混合检索/重排/父子 |
| 答案超出资料范围地“发挥” | 生成未遵守 grounding | 强化 prompt + 温度调低 |
| 答案引用来源对不上 | 模型乱标来源 | Citation 验证（见下） |

In [3]:
# 知识点·真调说明：Generation Error —— 同样资料同样问题，有/无 grounding 约束，模型“发挥”差多少
fact = '星云智能客服支持公有云 SaaS 与私有化两种部署方式，私有化部署需联系销售开通，通常在 5~10 个工作日完成。'
q = '私有化部署大概多少钱？'
print('① 生成“没被绑在资料上”（无 grounding 约束）—— 为给客户一个交代，可能自行补个价格')
_llm_live(
    prompt='资料：%s\n\n客户问：%s\n请给出答复。客户想尽快拿到报价决定是否采购，尽量给出一个大概价格范围。' % (fact, q),
    system='你是星云的产品销售顾问，语气热情，直接给客户明确答复。',
    fallback='未配置 Key 的固定样例（观察：价格不在资料里，是补出来的）：\n'
             '私有化部署一般 30~50 万起步，具体按坐席数与定制需求报价，我可以帮您约销售出正式方案。',
    temperature=0.2,
)
print()
print('② 生成“被绑在资料上”（grounding 约束）—— 资料没给价格，就守住“需询价”')
_llm_live(
    prompt='资料：%s\n\n客户问：%s' % (fact, q),
    system='你是严谨的知识库客服。规则：只依据给定资料回答；资料没有价格信息就明确说“资料未提及价格，需联系销售报价”，'
           '绝不自己编造任何数字或承诺。',
    fallback='未配置 Key 的固定样例：\n'
             '资料提到私有化部署需联系销售开通，但没有给出价格——具体费用要由销售按坐席数和定制需求报价。',
    temperature=0.1,
)
print('同一份资料、同一个问题，差的只是 system prompt 里有没有“禁止越出资料”的 grounding 约束。')
print('→ 对应三级定位里的 Generation Error（召回了却没按资料说）；治理 = grounding prompt + 低温，见第 2 节。')

① 生成“没被绑在资料上”（无 grounding 约束）—— 为给客户一个交代，可能自行补个价格
—— 模型实时输出 ——
您好！感谢您对星云智能客服的关注～😊

私有化部署的费用会根据您的具体需求（如并发坐席数、接入渠道数量、定制化程度、数据安全等级等）灵活配置，但为帮您快速决策，我们给您一个**典型中型客户（50坐席以内、标准功能+基础定制+3年服务）的参考范围：80万～150万元**（含首年实施、部署、培训及维保）。

✅ 如果您能简单告诉我：
- 预计坐席规模（比如30人？100人？）  
- 是否需要对接现有系统（如CRM、工单系统）？  
- 对数据合规是否有特殊要求（如等保三级、信创适配）？

我马上为您生成一份**精准报价单+部署排期表**，最快今天就能发您邮箱！🚀  
需要我立刻帮您启动吗？

② 生成“被绑在资料上”（grounding 约束）—— 资料没给价格，就守住“需询价”
—— 模型实时输出 ——
资料未提及价格，需联系销售报价。
同一份资料、同一个问题，差的只是 system prompt 里有没有“禁止越出资料”的 grounding 约束。
→ 对应三级定位里的 Generation Error（召回了却没按资料说）；治理 = grounding prompt + 低温，见第 2 节。


## 2. Grounded Generation：让答案落在资料上

- **Faithfulness（忠实度）**：衡量“答案是否全部有上下文依据”，是 RAG 的核心质量指标（第 35 课评测）；
- 手段：严格的 grounding prompt（25 课）、检索片段必须进上下文、必要时让模型逐句说明依据。

In [4]:
# 知识点·真调说明：逐句自标注依据 —— 让模型边答边给“每句依据”，无依据的句子当场现形
print('让模型“边答边交依据”：每个论断句末标注 <依据：N>（N 对应资料编号），无资料支撑就写 <依据：无>。')
ctx2 = ['星云标准版 998 元/月，含自动应答。', '标准版按坐席计，5 个坐席起，超出需加购。']
_pro = ('资料：\n' + '\n'.join('[%d] %s' % (i + 1, t) for i, t in enumerate(ctx2))
        + '\n\n问题：标准版支持私有化部署吗？月费与坐席怎么算？\n请作答，'
          '并逐句在句末标注依据：该句来自资料哪条就写 <依据：N>；资料没覆盖的点要么不写、要么明确标注 <依据：无>，'
          '绝不编造。')
_llm_live(
    prompt=_pro,
    system='你是严谨的知识库问答助手，只依据资料作答；资料没覆盖的点就明说无法确认。',
    fallback='未配置 Key 的固定样例：\n'
             '标准版月费 998 元/月，含自动应答 <依据：1>。\n'
             '坐席方面，标准版 5 个坐席起、超出需加购 <依据：2>。\n'
             '关于私有化部署，资料未提及，无法确认 <依据：无>。',
    temperature=0.2,
)
print('→ 生成时强制“句句可溯源”，没依据的内容就被当场拦下——这是 Grounded Generation 中'
      '“逐句说明依据”的实操形态，也是 Faithfulness 可被自动化评测的基础（本小节后一个 cell 的 verify 骨架即该思路）。')

让模型“边答边交依据”：每个论断句末标注 <依据：N>（N 对应资料编号），无资料支撑就写 <依据：无>。
—— 模型实时输出 ——
标准版月费为998元/月 <依据：1>；  
标准版按坐席计费，起购5个坐席，超出部分需加购 <依据：2>；  
标准版是否支持私有化部署，资料中未提及 <依据：无>。
→ 生成时强制“句句可溯源”，没依据的内容就被当场拦下——这是 Grounded Generation 中“逐句说明依据”的实操形态，也是 Faithfulness 可被自动化评测的基础（本小节后一个 cell 的 verify 骨架即该思路）。


In [5]:
# 一个轻量级“基于来源的验证器”骨架：检查答案是否由提供的片段支撑
from dotenv import load_dotenv; load_dotenv()
import os, json
API_KEY = os.getenv('DASHSCOPE_API_KEY', '')

def verify_answer(question, contexts, answer):
    """让 LLM 判定答案每句话是否有据，输出 JSON 结论"""
    from dashscope import Generation
    prompt = (f'问题:{question}\n参考资料:\n'+'\n'.join(contexts)+f'\n\n回答:{answer}\n\n'
              '判断该回答是否全部可被参考资料支撑。只输出 JSON: '
              '{"supported": true/false, "unsupported_parts": ["..."]}')
    r = Generation.call(model='qwen-plus', messages=[{'role':'user','content':prompt}],
                        api_key=API_KEY, result_format='message')
    return r.output.choices[0].message.content

ctxs = ['星云客服机器人支持公有云与私有化两种部署方式']
good = '支持公有云与私有化两种部署方式。'
bad  = '支持公有云与私有化，且赠送 1 年免费算力。'   # 后半句编造

if API_KEY and '你的' not in API_KEY:
    print('好答案:', verify_answer('部署方式', ctxs, good))
    print('坏答案:', verify_answer('部署方式', ctxs, bad))
else:
    print('LLM 验证器：能指出“赠送1年免费算力”无据可查（配置 .env 后运行）。')

好答案: {"supported": true, "unsupported_parts": []}
坏答案: {"supported": false, "unsupported_parts": ["且赠送 1 年免费算力"]}


## 3. 幻觉治理清单

1. **召回层**：混合检索 + 重排，提高 Context 正确率；
2. **上下文层**：去噪、去重、按预算截断（23 课）；
3. **生成层**：grounding prompt、低温、Don't-Know 出口；
4. **验证层**：LLM 逐句校验 + Citation 校验（来源是否真对应）；
5. **评测层**：Faithfulness/Answer Relevance 长期盯防（35 课）。

## 小结

- 先定位问题出在哪一层（检索还是生成），再决定对策；
- Grounded Generation + Citation 校验是防幻觉的双保险；
- 用忠实度指标持续监控（见评测章节）。